# Step 1 — Common 2021 LSOA harmonisation and provider assignment

**Purpose.** Place the 2001, 2011 and 2021 Census counts and registered-charity snapshots on the fixed set of 3,411 2021 LSOAs used throughout the dissertation.

**Reproducibility contract.** Historical additive counts are area-weighted first and rates are then recalculated. Source totals must be conserved. Charity registered-address coordinates are assigned to the same repaired 2021 polygons. This notebook does **not** build OD matrices or calculate E2SFCA.

Raw Census extracts, charity-level records and licensed boundary data are not distributed in this repository. Set `DISSERTATION_DATA_ROOT` to the private project root and optionally set `COMMON2021_OUTPUT_DIR` to an empty output directory.


## Analytical decisions frozen in Step 1

- Historical Census counts are allocated to the common 2021 geography before rates are recomputed; all source totals are conserved.
- Each eligible charity is assigned to a 2021 provider LSOA. Constant-2021-price income is transformed as \(\log(1+income)\) at charity level and then summed within provider LSOA \(j\) to define \(S_{jt}\). Missing income is excluded; recorded zero is retained.
- The same interior representative points and fixed December-2021 road graph are used for all three Census snapshots. Holding the network constant standardises temporal comparison; it does not reconstruct historical travel conditions.
- Off-network connectors are added at both route ends. Step 1 stores distances only; Step 2 applies the catchment and decay weights.

In [ ]:
from pathlib import Path
import hashlib
import os
import json
import platform
import sys
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT = Path(os.environ["DISSERTATION_DATA_ROOT"]).expanduser().resolve()
PACKAGE_DIR = ROOT / "final_data_and_analysis"
OUTPUT_ROOT = Path(
    os.environ.get("COMMON2021_OUTPUT_DIR", Path.cwd() / "_private_outputs" / "common2021")
).expanduser().resolve()
RUN_DIR = OUTPUT_ROOT / "qa"
UNIFIED_DIR = OUTPUT_ROOT / "unified-lsoa"
OUTPUT_DIR = RUN_DIR / "step1"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in (UNIFIED_DIR, OUTPUT_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

YEARS = (2001, 2011, 2021)
EXPECTED_NATIVE_LSOAS = {2001: 3230, 2011: 3285, 2021: 3411}
TARGET_LSOAS = 3411
LSOA_CODE = {year: f"LSOA{str(year)[-2:]}CD" for year in YEARS}
CHARITY_LSOA = {year: f"lsoa_{year}" for year in YEARS}
INCOME_COLUMN = {
    2001: "income_2021_gbp",
    2011: "income_2021_gbp",
    2021: "income_2021_gbp",
}

COVARIATE_PATH = {
    year: PACKAGE_DIR / "covariates" / f"{year}.csv" for year in YEARS
}
CHARITY_PATH = {
    year: PACKAGE_DIR / "Data_Spine" / "charity_rebuild_v2" / "07_final_outputs"
    / f"{year}charity.csv" for year in YEARS
}
BOUNDARY_PATH = {
    year: ROOT / "分析历史" / "varaibles" / "LSOA_boundaries" / "South_West_clipped"
    / f"LSOA_{year}_South_West.geojson"
    for year in YEARS
}
COMMON_BOUNDARY_GPKG = UNIFIED_DIR / "common2021_lsoa_spatial_foundation.gpkg"
COMMON_POINTS_PATH = UNIFIED_DIR / "common2021_lsoa_representative_points.csv"
COUNT_FIELDS = [
    "care50_num", "population_5plus", "total_population", "population_75plus",
    "population_50_64", "activity_limited_num", "tenure_households_total",
    "social_rented_num", "car_availability_households_total", "no_car_num",
    "household_composition_total", "one_person_num",
]
RATE_FORMULAS = {
    "care50_rate": ("care50_num", "population_5plus"),
    "population_75plus_rate": ("population_75plus", "total_population"),
    "population_50_64_rate": ("population_50_64", "total_population"),
    "activity_limited_rate": ("activity_limited_num", "total_population"),
    "social_rented_rate": ("social_rented_num", "tenure_households_total"),
    "no_car_rate": ("no_car_num", "car_availability_households_total"),
    "one_person_rate": ("one_person_num", "household_composition_total"),
}
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR)

In [ ]:
# Fail early if any frozen source is unavailable.
required_paths = [
    *COVARIATE_PATH.values(),
    *CHARITY_PATH.values(),
    *BOUNDARY_PATH.values(),
]
missing = [str(path) for path in required_paths if not path.is_file()]
assert not missing, f"Missing required inputs: {missing}"

source_inventory = pd.DataFrame(
    {
        "path": [str(path) for path in required_paths],
        "size_bytes": [path.stat().st_size for path in required_paths],
    }
)
display(source_inventory)

## 1. Repair native boundaries and build the common-2021 geography

The clipped source files contain a small number of geometry collections. After deterministic repair, 2001 and 2011 polygons are intersected with the 2021 target layer. Each source LSOA's positive intersection areas are normalised to sum to one. All Census counts are allocated with those weights, target rates are recomputed from their harmonised numerators and denominators, and regional totals must be conserved to floating-point tolerance. The 2021 table remains on its native geography through an identity crosswalk.

In [ ]:
def repair_lsoa_layer(year, boundary_path, analytical_codes):
    code_column = LSOA_CODE[year]
    raw = gpd.read_file(boundary_path).to_crs("EPSG:27700")
    assert code_column in raw.columns
    raw[code_column] = raw[code_column].astype(str).str.strip()

    selected = raw.loc[raw[code_column].isin(analytical_codes), [code_column, "geometry"]].copy()
    audit = {
        "year": year,
        "raw_rows": len(raw),
        "selected_rows": len(selected),
        "invalid_before": int((~selected.geometry.is_valid).sum()),
        "geometry_collections_before": int(selected.geom_type.eq("GeometryCollection").sum()),
        "non_polygon_before": int((~selected.geom_type.isin(["Polygon", "MultiPolygon"])).sum()),
    }

    selected = selected.loc[selected.geometry.notna() & ~selected.geometry.is_empty].copy()
    invalid = ~selected.geometry.is_valid
    if invalid.any():
        selected.loc[invalid, "geometry"] = selected.loc[invalid, "geometry"].make_valid()

    selected = selected.explode(index_parts=False, ignore_index=True)
    selected = selected.loc[selected.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    selected = selected.dissolve(by=code_column, as_index=False, aggfunc="first")
    selected = selected.rename(columns={code_column: "lsoa_code"})[["lsoa_code", "geometry"]]
    selected["lsoa_code"] = selected["lsoa_code"].astype(str).str.strip()
    selected = selected.sort_values("lsoa_code").reset_index(drop=True)

    assert selected["lsoa_code"].is_unique
    assert set(selected["lsoa_code"]) == set(analytical_codes)
    assert selected.geometry.notna().all() and (~selected.geometry.is_empty).all()
    assert selected.geometry.is_valid.all()
    assert selected.geom_type.isin(["Polygon", "MultiPolygon"]).all()

    audit.update(
        {
            "clean_rows": len(selected),
            "invalid_after": int((~selected.geometry.is_valid).sum()),
            "non_polygon_after": int((~selected.geom_type.isin(["Polygon", "MultiPolygon"])).sum()),
        }
    )
    return selected, audit


native_covariates = {}
native_polygons = {}
boundary_audit_rows = []

for year in YEARS:
    cov = pd.read_csv(COVARIATE_PATH[year], low_memory=False)
    assert len(cov) == EXPECTED_NATIVE_LSOAS[year]
    assert cov["lsoa_code"].is_unique
    assert cov["lsoa_code"].notna().all()
    cov["lsoa_code"] = cov["lsoa_code"].astype(str).str.strip()
    for field in COUNT_FIELDS:
        cov[field] = pd.to_numeric(cov[field], errors="raise").astype(float)
        assert cov[field].ge(0).all()

    geometry, audit = repair_lsoa_layer(year, BOUNDARY_PATH[year], cov["lsoa_code"])
    assert len(geometry) == EXPECTED_NATIVE_LSOAS[year]
    native_covariates[year] = cov
    native_polygons[year] = geometry
    boundary_audit_rows.append(audit)

boundary_audit = pd.DataFrame(boundary_audit_rows)
boundary_audit.to_csv(TABLE_DIR / "boundary_repair_audit.csv", index=False)
display(boundary_audit)

target_base = native_covariates[2021][
    ["lsoa_code", "lsoa_name", "ICB23CD", "ICB23NM", "rural_binary"]
].copy()
target_geometry = target_base[["lsoa_code", "lsoa_name"]].merge(
    native_polygons[2021], on="lsoa_code", validate="one_to_one"
)
target_geometry = gpd.GeoDataFrame(target_geometry, geometry="geometry", crs="EPSG:27700")
assert len(target_geometry) == TARGET_LSOAS
assert target_geometry["lsoa_code"].is_unique


def build_areal_crosswalk(year):
    target = target_geometry[["lsoa_code", "geometry"]].rename(
        columns={"lsoa_code": "target_lsoa_code"}
    )
    if year == 2021:
        areas = target.geometry.area.to_numpy(dtype=float)
        return pd.DataFrame({
            "source_lsoa_code": target["target_lsoa_code"].to_numpy(),
            "target_lsoa_code": target["target_lsoa_code"].to_numpy(),
            "intersection_area_m2": areas,
            "source_overlap_area_m2": areas,
            "source_normalized_weight": np.ones(TARGET_LSOAS, dtype=float),
        })

    source = native_polygons[year].rename(columns={"lsoa_code": "source_lsoa_code"})
    intersections = gpd.overlay(source, target, how="intersection", keep_geom_type=True)
    intersections = intersections.loc[
        intersections.geometry.notna() & ~intersections.geometry.is_empty
    ].copy()
    intersections["intersection_area_m2"] = intersections.geometry.area
    intersections = intersections.loc[intersections["intersection_area_m2"].gt(0)].copy()
    intersections["source_overlap_area_m2"] = intersections.groupby("source_lsoa_code")[
        "intersection_area_m2"
    ].transform("sum")
    intersections["source_normalized_weight"] = (
        intersections["intersection_area_m2"] / intersections["source_overlap_area_m2"]
    )
    crosswalk = intersections[[
        "source_lsoa_code", "target_lsoa_code", "intersection_area_m2",
        "source_overlap_area_m2", "source_normalized_weight",
    ]].copy()
    assert set(crosswalk["source_lsoa_code"]) == set(native_covariates[year]["lsoa_code"])
    sums = crosswalk.groupby("source_lsoa_code")["source_normalized_weight"].sum()
    assert np.allclose(sums, 1.0, atol=1e-12, rtol=0)
    return crosswalk


def harmonise_counts(year, crosswalk):
    source = native_covariates[year][["lsoa_code", *COUNT_FIELDS]].rename(
        columns={"lsoa_code": "source_lsoa_code"}
    )
    allocated = crosswalk.merge(source, on="source_lsoa_code", validate="many_to_one")
    for field in COUNT_FIELDS:
        allocated[field] = allocated[field] * allocated["source_normalized_weight"]
    totals = allocated.groupby("target_lsoa_code", as_index=False)[COUNT_FIELDS].sum().rename(
        columns={"target_lsoa_code": "lsoa_code"}
    )
    frame = target_base.merge(totals, on="lsoa_code", how="left", validate="one_to_one")
    frame.insert(0, "year", year)
    assert len(frame) == TARGET_LSOAS and frame["lsoa_code"].is_unique
    assert frame[COUNT_FIELDS].notna().all().all()
    for rate, (numerator, denominator) in RATE_FORMULAS.items():
        assert frame[denominator].gt(0).all()
        frame[rate] = frame[numerator] / frame[denominator]
    area_km2 = target_geometry.set_index("lsoa_code").geometry.area / 1_000_000
    frame["population_density_per_km2"] = (
        frame["total_population"] / frame["lsoa_code"].map(area_km2)
    )
    frame["rural_binary"] = pd.to_numeric(frame["rural_binary"], errors="raise").astype(int)
    ordered = [
        "year", "lsoa_code", "lsoa_name", "ICB23CD", "ICB23NM", *COUNT_FIELDS,
        *RATE_FORMULAS.keys(), "population_density_per_km2", "rural_binary",
    ]
    return frame[ordered]


crosswalks = {}
covariates = {}
conservation_rows = []
for year in YEARS:
    crosswalks[year] = build_areal_crosswalk(year)
    crosswalks[year].to_csv(
        UNIFIED_DIR / f"native_to_2021_areal_crosswalk_{year}.csv", index=False
    )
    covariates[year] = harmonise_counts(year, crosswalks[year])
    covariates[year].to_csv(
        UNIFIED_DIR / f"common2021_covariates_{year}.csv",
        index=False, encoding="utf-8-sig", float_format="%.15g",
    )
    for field in COUNT_FIELDS:
        native_total = float(native_covariates[year][field].sum())
        harmonised_total = float(covariates[year][field].sum())
        signed_difference = harmonised_total - native_total
        conservation_rows.append({
            "year": year,
            "variable": field,
            "native_lsoas": len(native_covariates[year]),
            "harmonised_lsoas": len(covariates[year]),
            "native_total": native_total,
            "harmonised_total": harmonised_total,
            "signed_difference": signed_difference,
            "absolute_difference": abs(signed_difference),
            "relative_error": abs(signed_difference) / max(abs(native_total), 1.0),
            "conservation_pass": abs(signed_difference) / max(abs(native_total), 1.0) < 1e-12,
        })

conservation_audit = pd.DataFrame(conservation_rows)
assert len(conservation_audit) == len(YEARS) * len(COUNT_FIELDS)
assert conservation_audit["conservation_pass"].all()
conservation_audit.to_csv(
    UNIFIED_DIR / "harmonisation_conservation_audit.csv", index=False
)
pd.concat([covariates[y] for y in YEARS], ignore_index=True).to_csv(
    UNIFIED_DIR / "common2021_covariates_long.csv",
    index=False, encoding="utf-8-sig", float_format="%.15g",
)

points_2021 = target_geometry.copy()
points_2021["geometry"] = points_2021.representative_point()
points_2021["easting"] = points_2021.geometry.x
points_2021["northing"] = points_2021.geometry.y
points_2021.drop(columns="geometry").to_csv(COMMON_POINTS_PATH, index=False)

if COMMON_BOUNDARY_GPKG.exists():
    COMMON_BOUNDARY_GPKG.unlink()
target_geometry.to_file(COMMON_BOUNDARY_GPKG, layer="lsoa_2021", driver="GPKG")
points_2021.to_file(COMMON_BOUNDARY_GPKG, layer="representative_points_2021", driver="GPKG", mode="a")

common_geography_audit = pd.DataFrame([
    {
        "year": year,
        "rows": len(covariates[year]),
        "unique_lsoa_codes": covariates[year]["lsoa_code"].nunique(),
        "missing_lsoa_codes": len(set(target_base["lsoa_code"]) - set(covariates[year]["lsoa_code"])),
        "duplicate_lsoa_codes": int(covariates[year]["lsoa_code"].duplicated().sum()),
        "same_target_code_set": set(covariates[year]["lsoa_code"]) == set(target_base["lsoa_code"]),
    }
    for year in YEARS
])
assert (common_geography_audit[["rows", "unique_lsoa_codes"]] == TARGET_LSOAS).all().all()
assert common_geography_audit["same_target_code_set"].all()
common_geography_audit.to_csv(TABLE_DIR / "common2021_geography_audit.csv", index=False)
display(conservation_audit)
display(common_geography_audit)
print("Canonical common-2021 spatial foundation:", COMMON_BOUNDARY_GPKG)

## 2. Assign charity snapshots to the common 2021 geography

Eligible registered-charity coordinates are spatially joined to repaired 2021 LSOA polygons. A deterministic nearest-polygon fallback handles boundary points. The native Census-year LSOA code remains a provenance field; the common-2021 assignment is the analytical provider geography.

Capacity is calculated only for records with usable income as `log(1 + income_2021_gbp)` and is then summed by provider LSOA. Registered addresses are treated as potential-access locations, not observed service-delivery sites.


In [ ]:
providers = {}
charity_audit_rows = []

for year in YEARS:
    charity = pd.read_csv(CHARITY_PATH[year], low_memory=False)
    lsoa_column = CHARITY_LSOA[year]
    income_column = INCOME_COLUMN[year]
    required = {
        "analysis_year", "charity_number", "postcode", "lat", "long",
        lsoa_column, income_column,
    }
    assert not (required - set(charity.columns)), f"{year}: missing {required - set(charity.columns)}"
    assert charity["charity_number"].is_unique
    assert charity["analysis_year"].eq(year).all()

    charity[lsoa_column] = charity[lsoa_column].astype("string").str.strip()
    charity[income_column] = pd.to_numeric(charity[income_column], errors="coerce")
    charity["lat"] = pd.to_numeric(charity["lat"], errors="coerce")
    charity["long"] = pd.to_numeric(charity["long"], errors="coerce")

    located = charity[lsoa_column].notna() & charity["lat"].notna() & charity["long"].notna()
    assert located.all(), f"{year}: charity location fields are incomplete"
    assert set(charity[lsoa_column]).issubset(set(native_covariates[year]["lsoa_code"]))
    assert charity[income_column].dropna().ge(0).all()

    charity_points = gpd.GeoDataFrame(
        charity.copy(),
        geometry=gpd.points_from_xy(charity["long"], charity["lat"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:27700")
    assigned = gpd.sjoin(
        charity_points,
        target_geometry[["lsoa_code", "geometry"]],
        how="left",
        predicate="within",
    ).rename(columns={"lsoa_code": "common2021_lsoa_code"})
    unmatched = assigned["common2021_lsoa_code"].isna()
    nearest_fallbacks = int(unmatched.sum())
    if unmatched.any():
        nearest = gpd.sjoin_nearest(
            charity_points.loc[unmatched, ["charity_number", "geometry"]],
            target_geometry[["lsoa_code", "geometry"]],
            how="left",
            distance_col="nearest_boundary_distance_m",
        ).rename(columns={"lsoa_code": "common2021_lsoa_code"})
        nearest = nearest.sort_values(
            ["charity_number", "nearest_boundary_distance_m", "common2021_lsoa_code"]
        ).drop_duplicates("charity_number")
        replacement = nearest.set_index("charity_number")["common2021_lsoa_code"]
        assigned.loc[unmatched, "common2021_lsoa_code"] = assigned.loc[
            unmatched, "charity_number"
        ].map(replacement)
    assert len(assigned) == len(charity)
    assert assigned["charity_number"].is_unique
    assert assigned["common2021_lsoa_code"].notna().all()
    assert set(assigned["common2021_lsoa_code"]).issubset(set(target_base["lsoa_code"]))
    # Existing year-specific LSOA codes are provenance fields. Coordinate-based
    # assignment to the repaired common-2021 polygons is the analytical rule;
    # any disagreement is retained in the audit below rather than overwritten.

    assignment_output = assigned.drop(columns=["geometry", "index_right"], errors="ignore").copy()
    assignment_output = assignment_output.rename(columns={lsoa_column: "native_lsoa_code"})
    assignment_output.to_csv(
        UNIFIED_DIR / f"common2021_charity_assignments_{year}.csv",
        index=False, encoding="utf-8-sig",
    )

    usable = assigned.loc[assigned[income_column].notna()].copy()
    usable["log1p_income_capacity"] = np.log1p(usable[income_column].to_numpy(dtype=float))
    supply = (
        usable.groupby("common2021_lsoa_code", as_index=False)
        .agg(
            registered_capacity_log1p_income=("log1p_income_capacity", "sum"),
            charity_records=("charity_number", "nunique"),
        )
        .rename(columns={"common2021_lsoa_code": "lsoa_code"})
        .sort_values("lsoa_code")
        .reset_index(drop=True)
    )
    provider = supply.merge(
        points_2021[["lsoa_code", "geometry"]], on="lsoa_code", validate="one_to_one"
    )
    provider = gpd.GeoDataFrame(provider, geometry="geometry", crs="EPSG:27700")
    assert provider["lsoa_code"].is_unique

    providers[year] = provider
    provider_table = provider.copy()
    provider_table["easting"] = provider_table.geometry.x
    provider_table["northing"] = provider_table.geometry.y
    provider_table.drop(columns="geometry").to_csv(
        TABLE_DIR / f"provider_lsoa_index_{year}.csv", index=False
    )
    provider_table.drop(columns="geometry").to_csv(
        UNIFIED_DIR / f"common2021_provider_lsoa_index_{year}.csv", index=False
    )

    charity_audit_rows.append(
        {
            "year": year,
            "registered_charities": len(charity),
            "charities_with_usable_income": len(usable),
            "charities_missing_income": int(charity[income_column].isna().sum()),
            "charities_with_zero_income": int(charity[income_column].eq(0).sum()),
            "provider_lsoas": len(provider),
            "sum_log1p_income_capacity": float(provider["registered_capacity_log1p_income"].sum()),
            "postcodes": charity["postcode"].nunique(),
            "nearest_polygon_fallbacks": nearest_fallbacks,
            "native_to_2021_lsoa_code_changes": int(
                assigned["common2021_lsoa_code"].ne(assigned[lsoa_column]).sum()
            ),
        }
    )

charity_audit = pd.DataFrame(charity_audit_rows)
charity_audit.to_csv(TABLE_DIR / "charity_provider_audit.csv", index=False)
display(charity_audit)

## 3. Validate the hand-off to the accessibility workflow

The final checks confirm the fixed 3,411-LSOA code set, count conservation, unique charity assignments and the expected final eligible-charity counts. The resulting manifest records only aggregate diagnostics; charity-level outputs remain private.


In [ ]:
EXPECTED_ELIGIBLE_CHARITIES = {2001: 2276, 2011: 3313, 2021: 3996}

assert conservation_audit["conservation_pass"].all()
assert common_geography_audit["rows"].eq(TARGET_LSOAS).all()
assert common_geography_audit["unique_lsoa_codes"].eq(TARGET_LSOAS).all()
assert charity_audit.set_index("year")["registered_charities"].to_dict() == EXPECTED_ELIGIBLE_CHARITIES

manifest = {
    "analysis": "common-2021 LSOA harmonisation and provider assignment",
    "years": list(YEARS),
    "target_lsoas": TARGET_LSOAS,
    "eligible_charities": EXPECTED_ELIGIBLE_CHARITIES,
    "count_fields_conserved": COUNT_FIELDS,
    "rate_rule": "allocate additive counts, then recalculate rates",
    "provider_capacity": "sum of charity-level log1p(income_2021_gbp) by common-2021 provider LSOA",
    "interpretation_boundary": "registered-address potential accessibility; not service delivery or observed travel",
    "outputs": {
        "unified_directory": str(UNIFIED_DIR.relative_to(OUTPUT_ROOT)),
        "qa_directory": str(RUN_DIR.relative_to(OUTPUT_ROOT)),
    },
}
manifest_path = RUN_DIR / "common2021_method_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("PASSED — common-2021 geography and provider assignments are ready for the E2SFCA workflows.")
display(charity_audit)


## Interpretation boundary

- Harmonised figures are descriptive area-allocated estimates on a common geography.
- Charity coordinates represent registered/contact addresses.
- No causal effect, observed journey, service use or unmet need is estimated in this step.
